## 🎯 Learning Objectives
* Understand the components of a modern CI/CD pipeline for containerized applications.
* Implement a Dockerfile for a Python-based RAG application.
* Design and configure a GitHub Actions workflow for automated testing, building, and deployment.
* Deploy a containerized application to a cloud platform (e.g., AWS ECS Fargate) using CI/CD.
* Securely manage cloud credentials within a CI/CD pipeline using OIDC.


# RAG03-L12: Exercise: Deploy Your Application with CI/CD

## Lesson Overview

In the previous lessons of RAG-03, you've built robust RAG applications using FastAPI for the backend, and either Streamlit or Next.js for the frontend, all containerized with Docker. Now, it's time to take these applications from local development to automated, production-ready deployment. This exercise challenges you to implement a Continuous Integration/Continuous Deployment (CI/CD) pipeline for a RAG application.

This lesson is crucial for full-stack developers and DevOps engineers aiming to streamline their development workflows and ensure reliable, repeatable deployments. We will focus on using GitHub Actions for CI/CD, deploying to AWS Elastic Container Service (ECS) with Fargate, and leveraging modern security practices like OpenID Connect (OIDC) for cloud authentication.

## Exercise Task

Your task is to establish a complete CI/CD pipeline for a simplified RAG application (or your own RAG application from previous lessons) using GitHub Actions to deploy to AWS ECS Fargate.

## Requirements

1.  **Application Setup**:
    *   Create a new GitHub repository for this exercise.
    *   Include a simple FastAPI application (provided as a starting point) that simulates a RAG backend.
    *   Provide a `requirements.txt` file listing all Python dependencies.
    *   Create a `Dockerfile` to containerize this FastAPI application.
    *   Include a basic set of unit tests for the FastAPI application using `pytest`.

2.  **AWS Infrastructure (Pre-requisite / Manual Setup)**:
    *   You will need an AWS account with appropriate permissions.
    *   Manually create an AWS Elastic Container Registry (ECR) repository to store your Docker images.
    *   Manually create an AWS ECS Cluster (Fargate launch type).
    *   Manually create an IAM Role and Policy for GitHub Actions to assume, allowing it to push to ECR and deploy to ECS. This role should be configured for OIDC trust with GitHub. (Detailed instructions for this setup are outside the scope of this exercise, but you should be familiar with the process from DevOps practices).

3.  **GitHub Actions Workflow (`.github/workflows/deploy.yml`)**:
    *   The workflow should be triggered on `push` events to the `main` branch.
    *   **CI Steps**:
        *   Checkout code.
        *   Set up Python environment.
        *   Install dependencies.
        *   Run unit tests (using `pytest`).
        *   Build the Docker image for your application.
    *   **CD Steps**:
        *   Configure AWS credentials using OIDC for secure authentication.
        *   Tag the Docker image with a version (e.g., Git SHA or timestamp).
        *   Push the Docker image to your designated AWS ECR repository.
        *   Create or update an AWS ECS Task Definition.
        *   Create or update an AWS ECS Service to deploy the new Task Definition to your ECS Cluster.
        *   Ensure the deployment waits for the service to stabilize.

## Evaluation Criteria

*   **Correctness of `Dockerfile`**: The Dockerfile should efficiently build a production-ready image for the FastAPI application.
*   **Completeness of `requirements.txt`**: All necessary dependencies are listed.
*   **Effectiveness of Unit Tests**: Basic tests are present and pass.
*   **GitHub Actions Workflow Logic**: The `deploy.yml` file correctly implements all CI/CD steps as described in the requirements.
*   **Secure AWS Authentication**: Proper use of OIDC for AWS credentials within GitHub Actions.
*   **Successful Automated Deployment**: The workflow successfully builds, pushes, and deploys the application to AWS ECS Fargate without manual intervention after a `git push`.
*   **Readability and Comments**: The workflow and code are well-structured and commented for clarity.


In [ ]:
# This cell provides a starting point for your RAG application and CI/CD setup.
# You should create these files in your GitHub repository.

# --- File: app.py (FastAPI application) ---
# This is a minimal FastAPI application to simulate a RAG backend.
# You can replace this with your actual RAG application from previous lessons.
app_py_content = """
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import os

app = FastAPI(
    title="Simple RAG Backend",
    description="A mock FastAPI application for CI/CD deployment exercise.",
    version="0.1.0"
)

class QueryRequest(BaseModel):
    query: str

class QueryResponse(BaseModel):
    response: str
    source: str

@app.get("/")
async def read_root():
    return {"message": "Welcome to the RAG Backend! Use /query for RAG operations."}

@app.post("/query", response_model=QueryResponse)
async def process_query(request: QueryRequest):
    # Simulate a RAG process
    if "hello" in request.query.lower():
        response = "Hello there! How can I assist you with RAG today?"
        source = "Internal Greeting System"
    elif "rag" in request.query.lower():
        response = f"RAG stands for Retrieval Augmented Generation. Your query was: '{request.query}'"
        source = "RAG Knowledge Base v1.0"
    else:
        response = f"I'm sorry, I don't have information on '{request.query}'. Please try another query."
        source = "Fallback Mechanism"
    
    return QueryResponse(response=response, source=source)

if __name__ == "__main__":
    # This block is for local development/testing, not typically used in Docker entrypoint
    port = int(os.environ.get("PORT", 8000))
    uvicorn.run(app, host="0.0.0.0", port=port)
"""

# --- File: requirements.txt ---
requirements_txt_content = """
fastapi==0.111.0
uvicorn[standard]==0.30.1
pydantic==2.7.1
pytest==8.2.1
"""

# --- File: Dockerfile ---
# This is a basic Dockerfile. You might want to enhance it with multi-stage builds
# for production efficiency.
dockerfile_content = """
# Use a lightweight Python base image
FROM python:3.11-slim-bookworm

# Set environment variables
ENV PYTHONUNBUFFERED 1
ENV APP_HOME /app

# Create and set the working directory
WORKDIR $APP_HOME

# Install system dependencies if any (e.g., for certain Python packages)
# RUN apt-get update && apt-get install -y --no-install-recommends \\
#     build-essential \\
#     && rm -rf /var/lib/apt/lists/*

# Copy requirements file and install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the application code
COPY . .

# Expose the port FastAPI runs on
EXPOSE 8000

# Command to run the application
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

# --- File: tests/test_app.py ---
# Basic unit tests for the FastAPI application.
test_app_py_content = """
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

def test_read_root():
    response = client.get("/")
    assert response.status_code == 200
    assert response.json() == {"message": "Welcome to the RAG Backend! Use /query for RAG operations."}

def test_process_query_hello():
    response = client.post("/query", json={"query": "Hello there"})
    assert response.status_code == 200
    assert "Hello there! How can I assist you with RAG today?" in response.json()["response"]
    assert response.json()["source"] == "Internal Greeting System"

def test_process_query_rag():
    response = client.post("/query", json={"query": "What is RAG?"})
    assert response.status_code == 200
    assert "Retrieval Augmented Generation" in response.json()["response"]
    assert response.json()["source"] == "RAG Knowledge Base v1.0"

def test_process_query_unknown():
    response = client.post("/query", json={"query": "Tell me about quantum physics"})
    assert response.status_code == 200
    assert "I'm sorry, I don't have information on 'Tell me about quantum physics'." in response.json()["response"]
    assert response.json()["source"] == "Fallback Mechanism"

def test_process_query_invalid_input():
    response = client.post("/query", json={"invalid_field": "test"})
    assert response.status_code == 422 # Unprocessable Entity due to Pydantic validation
"""

# --- File: .github/workflows/deploy.yml (GitHub Actions Workflow Skeleton) ---
# This is a skeleton for your GitHub Actions workflow.
# You will need to fill in the AWS-specific deployment steps and configure OIDC.
github_actions_skeleton = """
name: CI/CD to AWS ECS Fargate

on:
  push:
    branches:
      - main

env:
  AWS_REGION: us-east-1 # Replace with your desired AWS region
  ECR_REPOSITORY: your-ecr-repo-name # Replace with your ECR repository name
  ECS_CLUSTER_NAME: your-ecs-cluster-name # Replace with your ECS cluster name
  ECS_SERVICE_NAME: your-ecs-service-name # Replace with your ECS service name
  ECS_TASK_DEFINITION_FAMILY: your-ecs-task-definition-family # Replace with your ECS task definition family name
  CONTAINER_NAME: your-container-name # Name of the container in your task definition

permissions:
  id-token: write # Required for OIDC authentication
  contents: read # Required to checkout code

jobs:
  build-and-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Checkout code
      uses: actions/checkout@v4

    - name: Set up Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements.txt
        pip install pytest # Install pytest for running tests

    - name: Run unit tests
      run: pytest tests/

    - name: Configure AWS credentials
      uses: aws-actions/configure-aws-credentials@v4
      with:
        role-to-assume: arn:aws:iam::YOUR_AWS_ACCOUNT_ID:role/YOUR_GITHUB_ACTIONS_OIDC_ROLE # Replace with your IAM Role ARN
        aws-region: ${{ env.AWS_REGION }}

    - name: Login to Amazon ECR
      id: login-ecr
      uses: aws-actions/amazon-ecr-login@v2

    - name: Build, tag, and push Docker image to Amazon ECR
      id: build-image
      env:
        ECR_REGISTRY: ${{ steps.login-ecr.outputs.registry }}
        IMAGE_TAG: ${{ github.sha }}
      run: |
        docker build -t $ECR_REGISTRY/$ECR_REPOSITORY:$IMAGE_TAG .
        docker push $ECR_REGISTRY/$ECR_REPOSITORY:$IMAGE_TAG

    - name: Fill in the new image ID in the ECS task definition
      id: render-task-definition
      uses: aws-actions/amazon-ecs-render-task-definition@v1
      with:
        task-definition: .github/workflows/task-definition.json # You will need to create this file
        container-name: ${{ env.CONTAINER_NAME }}
        image: ${{ steps.login-ecr.outputs.registry }}/${{ env.ECR_REPOSITORY }}:${{ github.sha }}

    - name: Deploy Amazon ECS task definition
      uses: aws-actions/amazon-ecs-deploy-task-definition@v1
      with:
        task-definition: ${{ steps.render-task-definition.outputs.task-definition }}
        service: ${{ env.ECS_SERVICE_NAME }}
        cluster: ${{ env.ECS_CLUSTER_NAME }}
        wait-for-service-stability: true

    # Optional: Add steps for post-deployment smoke tests or notifications
"""

# --- File: .github/workflows/task-definition.json (ECS Task Definition Skeleton) ---
# This file defines your ECS Task Definition. You will need to customize it.
# You can get an initial version by exporting an existing task definition from AWS console.
ecs_task_definition_skeleton = """
{
  "family": "your-ecs-task-definition-family",
  "networkMode": "awsvpc",
  "containerDefinitions": [
    {
      "name": "your-container-name",
      "image": "YOUR_AWS_ACCOUNT_ID.dkr.ecr.YOUR_AWS_REGION.amazonaws.com/your-ecr-repo-name:latest",
      "cpu": 0,
      "portMappings": [
        {
          "containerPort": 8000,
          "hostPort": 8000,
          "protocol": "tcp"
        }
      ],
      "essential": true,
      "environment": [
        {
          "name": "PORT",
          "value": "8000"
        }
      ],
      "logConfiguration": {
        "logDriver": "awslogs",
        "options": {
          "awslogs-group": "/ecs/your-ecs-task-definition-family",
          "awslogs-region": "YOUR_AWS_REGION",
          "awslogs-stream-prefix": "ecs"
        }
      },
      "mountPoints": [],
      "volumesFrom": []
    }
  ],
  "requiresCompatibilities": [
    "FARGATE"
  ],
  "cpu": "256",
  "memory": "512",
  "executionRoleArn": "arn:aws:iam::YOUR_AWS_ACCOUNT_ID:role/ecsTaskExecutionRole",
  "taskRoleArn": "arn:aws:iam::YOUR_AWS_ACCOUNT_ID:role/ecsTaskRole"
}
"""

print("The following files are provided as a starting point. Create them in your GitHub repository:")
print("--- app.py ---")
print(app_py_content)
print("\n--- requirements.txt ---")
print(requirements_txt_content)
print("\n--- Dockerfile ---")
print(dockerfile_content)
print("\n--- tests/test_app.py ---")
print(test_app_py_content)
print("\n--- .github/workflows/deploy.yml (Skeleton) ---")
print(github_actions_skeleton)
print("\n--- .github/workflows/task-definition.json (Skeleton) ---")
print(ecs_task_definition_skeleton)

# Note: In a real Jupyter environment, you might use `%%writefile` magic command
# to create these files directly, but for this JSON output, we just print them.


## Your Turn: Implement the CI/CD Pipeline

Now it's your turn to complete the CI/CD pipeline.

**Steps to follow:**

1.  **Initialize your GitHub Repository**:
    *   Create a new public or private GitHub repository.
    *   Add the `app.py`, `requirements.txt`, `Dockerfile`, and `tests/test_app.py` files (from the setup cell) to your repository.
    *   Create the `.github/workflows/` directory and add the `deploy.yml` (skeleton) and `task-definition.json` (skeleton) files.

2.  **Configure AWS Resources**:
    *   **ECR Repository**: Create an ECR repository in your AWS account. Note its name.
    *   **ECS Cluster**: Create an ECS Cluster (Fargate launch type). Note its name.
    *   **ECS Task Definition**:
        *   Create an initial ECS Task Definition (you can do this via the AWS console, defining your container with the ECR image URI placeholder, port 8000, and appropriate CPU/Memory).
        *   Export this Task Definition as JSON.
        *   Update the `.github/workflows/task-definition.json` file with your exported JSON. **Crucially, ensure the `image` field in `containerDefinitions` is a placeholder that GitHub Actions can replace (e.g., `YOUR_AWS_ACCOUNT_ID.dkr.ecr.YOUR_AWS_REGION.amazonaws.com/your-ecr-repo-name:latest` or just `your-ecr-repo-name:latest` if the `aws-actions/amazon-ecs-render-task-definition` action handles the full URI).** The `aws-actions/amazon-ecs-render-task-definition` action expects the `image` field to be present and will replace it.
        *   Note the `family` name of your task definition and the `name` of your container within the task definition.
    *   **ECS Service**: Create an ECS Service associated with your cluster and task definition. Note its name.
    *   **IAM Role for GitHub Actions (OIDC)**:
        *   Create an IAM Role that GitHub Actions can assume.
        *   Configure a Trust Policy for this role to trust `token.actions.githubusercontent.com` with conditions matching your repository (e.g., `sub: repo:your-github-org/your-repo:ref:refs/heads/main`).
        *   Attach an IAM Policy to this role that grants permissions for:
            *   `ecr:GetDownloadUrlForLayer`, `ecr:BatchGetImage`, `ecr:BatchCheckLayerAvailability`, `ecr:PutImage`, `ecr:InitiateLayerUpload`, `ecr:UploadLayerPart`, `ecr:CompleteLayerUpload` (for ECR push).
            *   `ecs:DescribeServices`, `ecs:UpdateService`, `ecs:RegisterTaskDefinition`, `ecs:DescribeTaskDefinition`, `ecs:ListTaskDefinitions` (for ECS deployment).
            *   `iam:PassRole` (to pass the Task Execution Role and Task Role to ECS).
        *   Note the ARN of this IAM Role.

3.  **Update GitHub Actions Workflow (`.github/workflows/deploy.yml`)**:
    *   Fill in the `env` variables with your AWS region, ECR repository name, ECS cluster name, ECS service name, ECS task definition family name, and container name.
    *   Update the `role-to-assume` in the `configure-aws-credentials` step with the ARN of the IAM Role you created for GitHub Actions.
    *   Ensure the `task-definition.json` path is correct.

4.  **Commit and Push**:
    *   Commit all your changes to the `main` branch of your GitHub repository.
    *   Observe the GitHub Actions workflow execution. It should automatically build, test, push the Docker image, and deploy your application to AWS ECS Fargate.

5.  **Verify Deployment**:
    *   Check the AWS ECS console to ensure your service is running the new task definition.
    *   Access your deployed application (e.g., via the Load Balancer URL if configured) and test the `/` and `/query` endpoints.

Good luck! This exercise will solidify your understanding of modern, automated deployment practices.


In [ ]:
# This cell provides a complete, high-quality reference solution for the CI/CD pipeline.
# Remember to replace placeholders like YOUR_AWS_ACCOUNT_ID, YOUR_AWS_REGION,
# and specific resource names with your actual AWS environment details.

# --- File: app.py (FastAPI application - same as setup) ---
app_py_solution = """
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import os

app = FastAPI(
    title="Simple RAG Backend",
    description="A mock FastAPI application for CI/CD deployment exercise.",
    version="0.1.0"
)

class QueryRequest(BaseModel):
    query: str

class QueryResponse(BaseModel):
    response: str
    source: str

@app.get("/")
async def read_root():
    return {"message": "Welcome to the RAG Backend! Use /query for RAG operations."}

@app.post("/query", response_model=QueryResponse)
async def process_query(request: QueryRequest):
    # Simulate a RAG process
    if "hello" in request.query.lower():
        response = "Hello there! How can I assist you with RAG today?"
        source = "Internal Greeting System"
    elif "rag" in request.query.lower():
        response = f"RAG stands for Retrieval Augmented Generation. Your query was: '{request.query}'"
        source = "RAG Knowledge Base v1.0"
    else:
        response = f"I'm sorry, I don't have information on '{request.query}'. Please try another query."
        source = "Fallback Mechanism"
    
    return QueryResponse(response=response, source=source)

if __name__ == "__main__":
    port = int(os.environ.get("PORT", 8000))
    uvicorn.run(app, host="0.0.0.0", port=port)
"""

# --- File: requirements.txt (same as setup) ---
requirements_txt_solution = """
fastapi==0.111.0
uvicorn[standard]==0.30.1
pydantic==2.7.1
pytest==8.2.1
"""

# --- File: Dockerfile (Enhanced with multi-stage build for production) ---
dockerfile_solution = """
# Stage 1: Builder - Install dependencies
FROM python:3.11-slim-bookworm as builder

# Set environment variables
ENV PYTHONUNBUFFERED 1
ENV APP_HOME /app

# Create and set the working directory
WORKDIR $APP_HOME

# Install system dependencies if any (e.g., for certain Python packages)
# RUN apt-get update && apt-get install -y --no-install-recommends \\
#     build-essential \\
#     && rm -rf /var/lib/apt/lists/*

# Copy requirements file and install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Stage 2: Production - Copy only necessary files
FROM python:3.11-slim-bookworm

# Set environment variables
ENV PYTHONUNBUFFERED 1
ENV APP_HOME /app

# Create and set the working directory
WORKDIR $APP_HOME

# Copy installed dependencies from the builder stage
COPY --from=builder /usr/local/lib/python3.11/site-packages /usr/local/lib/python3.11/site-packages
COPY --from=builder /usr/local/bin/uvicorn /usr/local/bin/uvicorn # Copy uvicorn executable

# Copy the application code
COPY . .

# Expose the port FastAPI runs on
EXPOSE 8000

# Command to run the application
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

# --- File: tests/test_app.py (same as setup) ---
test_app_py_solution = """
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

def test_read_root():
    response = client.get("/")
    assert response.status_code == 200
    assert response.json() == {"message": "Welcome to the RAG Backend! Use /query for RAG operations."}

def test_process_query_hello():
    response = client.post("/query", json={"query": "Hello there"})
    assert response.status_code == 200
    assert "Hello there! How can I assist you with RAG today?" in response.json()["response"]
    assert response.json()["source"] == "Internal Greeting System"

def test_process_query_rag():
    response = client.post("/query", json={"query": "What is RAG?"})
    assert response.status_code == 200
    assert "Retrieval Augmented Generation" in response.json()["response"]
    assert response.json()["source"] == "RAG Knowledge Base v1.0"

def test_process_query_unknown():
    response = client.post("/query", json={"query": "Tell me about quantum physics"})
    assert response.status_code == 200
    assert "I'm sorry, I don't have information on 'Tell me about quantum physics'." in response.json()["response"]
    assert response.json()["source"] == "Fallback Mechanism"

def test_process_query_invalid_input():
    response = client.post("/query", json={"invalid_field": "test"})
    assert response.status_code == 422 # Unprocessable Entity due to Pydantic validation
"""

# --- File: .github/workflows/deploy.yml (Complete Solution) ---
github_actions_solution = """
name: CI/CD to AWS ECS Fargate

on:
  push:
    branches:
      - main # Trigger on push to the main branch

env:
  # --- IMPORTANT: Replace these placeholders with your actual AWS resource names ---
  AWS_REGION: us-east-1 # e.g., us-east-1, eu-west-1
  ECR_REPOSITORY: rag-app-backend # Your ECR repository name
  ECS_CLUSTER_NAME: rag-app-cluster # Your ECS cluster name
  ECS_SERVICE_NAME: rag-app-service # Your ECS service name
  ECS_TASK_DEFINITION_FAMILY: rag-app-task-definition # The family name of your ECS task definition
  CONTAINER_NAME: rag-app-container # The name of your container within the task definition

permissions:
  id-token: write # Required for OIDC authentication with AWS
  contents: read # Required to checkout the repository code

jobs:
  build-and-deploy:
    runs-on: ubuntu-latest # Use a fresh Ubuntu runner for each job

    steps:
    - name: Checkout code
      uses: actions/checkout@v4 # Action to checkout your repository code

    - name: Set up Python
      uses: actions/setup-python@v5 # Action to set up Python environment
      with:
        python-version: '3.11' # Specify Python version

    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip # Upgrade pip
        pip install -r requirements.txt # Install application dependencies
        pip install pytest # Install pytest for running tests
      # Cache pip dependencies for faster builds (optional but recommended for real projects)
      # uses: actions/cache@v4
      # with:
      #   path: ~/.cache/pip
      #   key: ${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}
      #   restore-keys: |
      #     ${{ runner.os }}-pip-

    - name: Run unit tests
      run: pytest tests/ # Execute pytest tests

    - name: Configure AWS credentials
      uses: aws-actions/configure-aws-credentials@v4 # Action to configure AWS credentials using OIDC
      with:
        # --- IMPORTANT: Replace with the ARN of your IAM Role for GitHub Actions ---
        # This role must have a trust policy configured for GitHub's OIDC provider
        # and permissions to push to ECR and deploy to ECS.
        role-to-assume: arn:aws:iam::YOUR_AWS_ACCOUNT_ID:role/GitHubActionsOIDC-RAGAppDeployRole
        aws-region: ${{ env.AWS_REGION }}

    - name: Login to Amazon ECR
      id: login-ecr
      uses: aws-actions/amazon-ecr-login@v2 # Action to log in to ECR

    - name: Build, tag, and push Docker image to Amazon ECR
      id: build-image
      env:
        ECR_REGISTRY: ${{ steps.login-ecr.outputs.registry }} # ECR registry URI
        IMAGE_TAG: ${{ github.sha }} # Use Git SHA as image tag for uniqueness
      run: |
        # Build the Docker image
        docker build -t $ECR_REGISTRY/$ECR_REPOSITORY:$IMAGE_TAG .
        # Push the Docker image to ECR
        docker push $ECR_REGISTRY/$ECR_REPOSITORY:$IMAGE_TAG
        echo "Image pushed: $ECR_REGISTRY/$ECR_REPOSITORY:$IMAGE_TAG"

    - name: Fill in the new image ID in the ECS task definition
      id: render-task-definition
      uses: aws-actions/amazon-ecs-render-task-definition@v1 # Action to update task definition with new image
      with:
        task-definition: .github/workflows/task-definition.json # Path to the task definition file
        container-name: ${{ env.CONTAINER_NAME }} # Name of the container to update
        image: ${{ steps.login-ecr.outputs.registry }}/${{ env.ECR_REPOSITORY }}:${{ github.sha }} # New image URI

    - name: Deploy Amazon ECS task definition
      uses: aws-actions/amazon-ecs-deploy-task-definition@v1 # Action to deploy the updated task definition
      with:
        task-definition: ${{ steps.render-task-definition.outputs.task-definition }} # The rendered task definition
        service: ${{ env.ECS_SERVICE_NAME }} # The ECS service to update
        cluster: ${{ env.ECS_CLUSTER_NAME }} # The ECS cluster
        wait-for-service-stability: true # Wait for the service to stabilize after deployment

    - name: Post-deployment Smoke Test (Optional)
      run: |
        # In a real scenario, you would query your load balancer or service endpoint
        # to ensure the deployed application is healthy and responsive.
        # Example: curl -f http://YOUR_LOAD_BALANCER_DNS/
        echo "Deployment successful. Perform manual or automated smoke tests."
"""

# --- File: .github/workflows/task-definition.json (Example Task Definition) ---
# This is an example. You should export your actual task definition from AWS.
# Key fields to note:
# - "family": Must match ECS_TASK_DEFINITION_FAMILY in deploy.yml
# - "containerDefinitions": Array of containers. The "name" here must match CONTAINER_NAME.
# - "image": This will be replaced by the GitHub Action.
# - "portMappings": Ensure containerPort matches the EXPOSE in Dockerfile (8000).
# - "cpu", "memory": Fargate requires these to be defined.
# - "executionRoleArn", "taskRoleArn": Replace with your actual IAM role ARNs.
ecs_task_definition_solution = """
{
  "family": "rag-app-task-definition",
  "networkMode": "awsvpc",
  "containerDefinitions": [
    {
      "name": "rag-app-container",
      "image": "YOUR_AWS_ACCOUNT_ID.dkr.ecr.YOUR_AWS_REGION.amazonaws.com/rag-app-backend:latest",
      "cpu": 0,
      "portMappings": [
        {
          "containerPort": 8000,
          "hostPort": 8000,
          "protocol": "tcp"
        }
      ],
      "essential": true,
      "environment": [
        {
          "name": "PORT",
          "value": "8000"
        }
      ],
      "logConfiguration": {
        "logDriver": "awslogs",
        "options": {
          "awslogs-group": "/ecs/rag-app-task-definition",
          "awslogs-region": "YOUR_AWS_REGION",
          "awslogs-stream-prefix": "ecs"
        }
      },
      "mountPoints": [],
      "volumesFrom": []
    }
  ],
  "requiresCompatibilities": [
    "FARGATE"
  ],
  "cpu": "256",
  "memory": "512",
  "executionRoleArn": "arn:aws:iam::YOUR_AWS_ACCOUNT_ID:role/ecsTaskExecutionRole",
  "taskRoleArn": "arn:aws:iam::YOUR_AWS_ACCOUNT_ID:role/ecsTaskRole"
}
"""

print("Below are the complete solution files. Remember to customize placeholders for your AWS environment.")
print("\n--- app.py ---")
print(app_py_solution)
print("\n--- requirements.txt ---")
print(requirements_txt_solution)
print("\n--- Dockerfile (Multi-stage) ---")
print(dockerfile_solution)
print("\n--- tests/test_app.py ---")
print(test_app_py_solution)
print("\n--- .github/workflows/deploy.yml (Complete) ---")
print(github_actions_solution)
print("\n--- .github/workflows/task-definition.json (Example) ---")
print(ecs_task_definition_solution)
